In [ ]:
from tqdm import tqdm
from model import CNNTimeSeriesClassifier, ImprovedCustomDataset, train_model,load_model,convert_data
import pandas as pd
import torch
import json
import numpy as np

model_path = r"C:\SKB_co_Project\Mocap\git\ALL_mocap\model\src\asset\model\model_88.pt"
with open(r"C:\SKB_co_Project\Mocap\git\ALL_mocap\model\src\asset\labels.json","r",encoding='utf-8') as f:
        label = json.load(f)
with open(r"C:\SKB_co_Project\Mocap\git\ALL_mocap\model\src\asset\rollback.json","r",encoding='utf-8') as f:
        content = json.load(f)

def padding(features):
    if len(features) >= 50:
        # If longer than chunk_size, use uniform sampling
        indices = np.linspace(0, len(features)-1, 50, dtype=int)
        sequence = features[indices]
    else:
        # If shorter, pad with zeros at the end
        sequence = np.zeros((50, features.shape[1]))
        sequence[:len(features)] = features
    return sequence


model_r = CNNTimeSeriesClassifier((50,28),51)
# model = torch.load(r"F:\Hybridmodel-project\Sign_Language_Detection\model_2\finetuned_all_gestures_20251012_011755.pth",weights_only=False)
# model_r.load_state_dict(model)
# model_r(torch.rand(1,50,28))

        
if torch.cuda.is_available():
    # model = torch.load(rf"{model_path}",weights_only=False)
    # model_r.load_state_dict(model)
    # model_r.to("cuda")
    # device = "cuda"
    
    model_r = torch.load(model_path,weights_only=False)
    model_r.to("cuda")
else:
    model = torch.load(rf"{model_path}",weights_only=False,map_location=torch.device('cpu'))
    model_r.load_state_dict(model)
    device = "cpu"
    model_r = torch.load(model_path,weights_only=False)
    
    
    # model
model_r.double()
model_r.eval()


# files = glob.glob(r"./collect_data/out_long_word/out_long_word/*")
files = [r"C:\SKB_co_Project\Mocap\git\ALL_mocap\model\src\asset\other\eval_data.csv"]
# files = [r"F:\Hybridmodel-project\Sign_Language_Detection\dump\20250715_111750_DATA_INDICATOR_sensor.csv"]
for i in files:
    df = pd.read_csv(i)
    df["Label"] = df["Label"].apply(lambda x:x.replace("'",""))


    df['group_id'] = (df['Label'] != df['Label'].shift()).cumsum()
    grouped_data = [(group.drop('group_id', axis=1).values[:,1:-1],group.Label.value_counts().index[0]) for _, group in df.groupby('group_id')]

    y_pred = []
    y_true = []
    prob_x = []
    prob_y = []
    y_pred_text = []
    with torch.no_grad():
        for i in tqdm(range(len(grouped_data))):
            data = torch.from_numpy(padding(grouped_data[i][0]).astype(np.float64)).unsqueeze(0)
            output = model_r(data.double().to("cuda"))
            prob_x.append(output)
            prob_y.append(grouped_data[i][1])
            if grouped_data[i][1] !="nothing" and grouped_data[i][1] !="break_time":
                text = content[str(torch.argmax(output).item())]
                y_pred.append(int(torch.argmax(output).item()))
                y_pred_text.append(text)
                y_true.append(label[grouped_data[i][1]])
                # print(text,"real text:",grouped_data[i][1])  
    print("\n\n")
    
    
from sklearn.metrics import f1_score,recall_score,accuracy_score,confusion_matrix,ConfusionMatrixDisplay
f1_scores = f1_score(y_true, y_pred, average="micro")
print("f1 score    ",f1_scores)
recall_scores = recall_score(y_true, y_pred, average="micro")
print("recal score ",recall_scores)
acc = accuracy_score(y_true, y_pred)
print("acc score   ",acc)
cnf = confusion_matrix(y_true, y_pred)

TypeError: Expected state_dict to be dict-like, got <class 'model.CNNTimeSeriesClassifier'>.